# NB08: Natural samples with OOD detection, QRF uncertainty, and H&C 2011 geotherm

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
)
import warnings
warnings.filterwarnings('ignore')

import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import IsolationForest
from quantile_forest import RandomForestQuantileRegressor

In [2]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [3]:
from src.geotherm import hasterok_chapman_geotherm


In [4]:
# Sanity-check the geotherm at fixed depths.
# H&C 2011 cold (40 mW/m^2) cratonic geotherm: T ~ 500-700 C at 100 km, ~ 850-1000 C at 200 km.
# Lithostatic pressure at 100 km: ~30 kbar (40 km crust @ 2800 + 60 km mantle @ 3300).
z_km = np.linspace(0, 250, 500)
T_40, P_40 = hasterok_chapman_geotherm(40, z_km)
T_60, P_60 = hasterok_chapman_geotherm(60, z_km)
i100 = np.argmin(np.abs(z_km - 100))
i200 = np.argmin(np.abs(z_km - 200))
print(f'40 mW/m^2: T(100 km)={T_40[i100]:.0f} C, P(100 km)={P_40[i100]:.1f} kbar')
print(f'40 mW/m^2: T(200 km)={T_40[i200]:.0f} C, P(200 km)={P_40[i200]:.1f} kbar')
print(f'60 mW/m^2: T(100 km)={T_60[i100]:.0f} C, P(100 km)={P_60[i100]:.1f} kbar')
assert 27 <= P_40[i100] <= 33, f'P at 100 km should be ~30 kbar, got {P_40[i100]:.1f}'
assert 400 <= T_40[i100] <= 800, f'T at 100 km on 40 mW/m^2 should be 400-800 C, got {T_40[i100]:.0f}'
assert 800 <= T_60[i100] <= 1300, f'T at 100 km on 60 mW/m^2 should be 800-1300 C, got {T_60[i100]:.0f}'
print('Geotherm sanity check: PASS')

40 mW/m^2: T(100 km)=518 C, P(100 km)=30.5 kbar
40 mW/m^2: T(200 km)=764 C, P(200 km)=62.8 kbar
60 mW/m^2: T(100 km)=1240 C, P(100 km)=30.5 kbar
Geotherm sanity check: PASS


In [5]:
# Override config path to match user's actual file
LIN2023_NATURAL = DATA_NATURAL / 'natural_opx_cleaned.csv'

# Hard check the natural sample file exists
if not LIN2023_NATURAL.exists():
    raise FileNotFoundError(
        f'Natural sample file not found: {LIN2023_NATURAL}\n'
        f'NB08 must not run on training data. Place a natural sample CSV at:\n'
        f'  {DATA_NATURAL}\n'
    )

# Try to load. The file may be xlsx mislabeled as csv.
def try_load_natural(path):
    try:
        return pd.read_csv(path), 'csv'
    except UnicodeDecodeError:
        pass
    except Exception:
        pass
    # Try as xlsx (any sheet)
    try:
        xl = pd.ExcelFile(path)
        return None, ('xlsx', xl.sheet_names)
    except Exception as e:
        raise IOError(f'Could not parse {path}: {e}')

result, kind = try_load_natural(LIN2023_NATURAL)
print(f'Natural file kind: {kind}')

Natural file kind: csv


In [6]:
# Load natural sample CSV directly (bypassing the legacy Excel sheet scraper)
df_raw = pd.read_csv(LIN2023_NATURAL)
print(f'Loaded {len(df_raw)} rows from {LIN2023_NATURAL.name}')
df_raw.columns = [str(c).strip() for c in df_raw.columns]

# Target column renaming map
rename = {}
targets = {'SiO2': 'SiO2', 'TiO2': 'TiO2', 'Al2O3': 'Al2O3', 'Cr2O3': 'Cr2O3',
           'FeO_total': 'FeO_total', 'FeO': 'FeO', 'Fe2O3': 'Fe2O3', 'MnO': 'MnO', 
           'MgO': 'MgO', 'CaO': 'CaO', 'Na2O': 'Na2O', 'K2O': 'K2O'}

for col in df_raw.columns:
    clow = col.lower().replace('_opx', '').replace('opx_', '').replace('(wt%)', '').strip()
    for tgt in targets:
        if clow == tgt.lower():
            rename[col] = tgt
            break

df_raw = df_raw.rename(columns=rename)
df_natural = df_raw[[c for c in targets if c in df_raw.columns]].copy()
df_natural = df_natural.apply(pd.to_numeric, errors='coerce')

# Ensure FeO_total exists
if 'FeO_total' not in df_natural.columns:
    df_natural['FeO_total'] = df_natural.get('FeO', 0).fillna(0) + 0.8998 * df_natural.get('Fe2O3', 0).fillna(0)

# Drop rows missing core oxides
n_before = len(df_natural)
df_natural = df_natural.dropna(subset=['SiO2', 'Al2O3', 'FeO_total', 'MgO', 'CaO'])
print(f'After core-oxide dropna: {len(df_natural)}/{n_before}')
print(f'df_natural shape: {df_natural.shape}')

Loaded 53050 rows from natural_opx_cleaned.csv


After core-oxide dropna: 53050/53050
df_natural shape: (53050, 9)


In [7]:
# Apply NB01-style cleaning to natural samples (only if real data is present)
if len(df_natural) > 0:
    ox_data = {
        'SiO2':  (60.084,   1, 2),
        'TiO2':  (79.866,   1, 2),
        'Al2O3': (101.961,  2, 3),
        'Cr2O3': (151.99,   2, 3),
        'FeO':   (71.844,   1, 1),
        'MnO':   (70.937,   1, 1),
        'MgO':   (40.304,   1, 1),
        'CaO':   (56.077,   1, 1),
        'Na2O':  (61.979,   2, 1),
        'K2O':   (94.195,   2, 1),
    }
    for ox in ox_data:
        if ox not in df_natural.columns:
            df_natural[ox] = 0.0
        df_natural[ox] = df_natural[ox].fillna(0)

    def cation_recalc(row):
        mol_cat, mol_ox = {}, 0.0
        for ox, (mw, n_cat, n_ox) in ox_data.items():
            wt = row['FeO_total'] if ox == 'FeO' else row[ox]
            wt = float(wt) if pd.notna(wt) else 0.0
            moles_ox_unit = wt / mw
            mol_cat[ox] = moles_ox_unit * n_cat
            mol_ox += moles_ox_unit * n_ox
        if mol_ox <= 0:
            return {k+'_cat': 0.0 for k in ox_data} | {'cation_sum': 0.0}
        f = 6.0 / mol_ox
        out = {ox+'_cat': mol_cat[ox]*f for ox in ox_data}
        out['cation_sum'] = sum(out.values())
        return out

    cat_df = pd.DataFrame([cation_recalc(r) for _, r in df_natural.iterrows()], index=df_natural.index)
    df_natural = pd.concat([df_natural, cat_df], axis=1)

    df_natural['oxide_total'] = df_natural[list(ox_data.keys())].sum(axis=1)
    n_before = len(df_natural)
    df_natural = df_natural[df_natural['oxide_total'].between(OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX)].copy()
    print(f'oxide-total filter: {n_before} -> {len(df_natural)}')
    df_natural = df_natural[df_natural['cation_sum'].between(CATION_SUM_MIN, CATION_SUM_MAX)].copy()
    print(f'cation-sum filter: {len(df_natural)}')

    mg, fe, ca = df_natural['MgO_cat'], df_natural['FeO_cat'], df_natural['CaO_cat']
    denom = (mg + fe + ca).replace(0, np.nan)
    df_natural['En_frac'] = 100 * mg / denom
    df_natural['Fs_frac'] = 100 * fe / denom
    df_natural['Wo_frac'] = 100 * ca / denom
    df_natural['Wo'] = df_natural['Wo_frac']
    df_natural = df_natural[df_natural['Wo'] <= WO_MAX_MOL_PCT].copy()
    print(f'pigeonite filter: {len(df_natural)}')

    df_natural['Mg_num'] = 100 * df_natural['MgO_cat'] / (df_natural['MgO_cat'] + df_natural['FeO_cat']).replace(0, np.nan)
    df_natural['Al_IV'] = (2.0 - df_natural['SiO2_cat']).clip(lower=0)
    df_natural['Al_VI'] = (df_natural['Al2O3_cat'] - df_natural['Al_IV']).clip(lower=0)
    df_natural['MgTs'] = np.minimum(df_natural['Al_IV'], df_natural['Al_VI']).clip(lower=0)
    print(f'After all NB01-style cleaning: {len(df_natural)}')
else:
    print('No natural samples to clean.')

oxide-total filter: 53050 -> 3237
cation-sum filter: 3208
pigeonite filter: 3145
After all NB01-style cleaning: 3145


In [8]:
def parse_params(s):
    import ast, json
    if isinstance(s, dict): return s
    try: return ast.literal_eval(s)
    except:
        try: return json.loads(s)
        except: return {}

# Identify the opx-only winning models for QRF + IsolationForest
multi = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
rf_opx = multi[(multi.track == 'opx_only') & 
               (multi.model_name == 'RF') & 
               (multi.split_seed == 42)]

rf_T_opx = rf_opx[rf_opx.target == 'T_C'].iloc[0]
rf_P_opx = rf_opx[rf_opx.target == 'P_kbar'].iloc[0]

fs_T_opx, fs_P_opx = rf_T_opx['feature_set'], rf_P_opx['feature_set']
params_T_opx = parse_params(rf_T_opx['best_params'])
params_P_opx = parse_params(rf_P_opx['best_params'])
print(f'RF opx-only: T uses {fs_T_opx}, P uses {fs_P_opx}')

# Load training data + index split for the opx-only track
df_opx_full = pd.read_parquet(DATA_PROC / 'opx_clean_opx_only.parquet')
idx_tr_opx = np.load(DATA_SPLITS / 'train_indices_opx.npy')
df_train_opx = df_opx_full.loc[idx_tr_opx].copy()
y_train_T = df_train_opx['T_C'].values
y_train_P = df_train_opx['P_kbar'].values

# Build training feature matrices for both targets (different feature sets possible)
X_train_T_opx, names_T_opx = build_feature_matrix(df_train_opx, fs_T_opx, use_liq=False)
X_train_P_opx, names_P_opx = build_feature_matrix(df_train_opx, fs_P_opx, use_liq=False)
print(f'X_train_T shape: {X_train_T_opx.shape}, X_train_P shape: {X_train_P_opx.shape}')

RF opx-only: T uses raw, P uses raw


X_train_T shape: (845, 16), X_train_P shape: (845, 16)


In [9]:
# Train opx-only QRFs (mirror the winning RF hyperparameters) and IsolationForest
qrf_T_opx = RandomForestQuantileRegressor(**params_T_opx, random_state=SEED_MODEL, n_jobs=-1)
qrf_T_opx.fit(X_train_T_opx, y_train_T)
qrf_P_opx = RandomForestQuantileRegressor(**params_P_opx, random_state=SEED_MODEL, n_jobs=-1)
qrf_P_opx.fit(X_train_P_opx, y_train_P)
joblib.dump(qrf_T_opx, MODELS / 'model_QRF_T_C_opx_only.joblib')
joblib.dump(qrf_P_opx, MODELS / 'model_QRF_P_kbar_opx_only.joblib')
print('Saved opx-only QRF models')

# IsolationForest fit on opx-only training features (use the P feature set for OOD)
iso = IsolationForest(n_estimators=200, contamination='auto', random_state=SEED_MODEL, n_jobs=-1)
iso.fit(X_train_P_opx)
joblib.dump(iso, MODELS / 'isolation_forest_opx_only.joblib')
print('Saved IsolationForest')

Saved opx-only QRF models


Saved IsolationForest


In [10]:
# Score natural samples (or generate empty results if no real natural data)
if len(df_natural) > 0:
    X_nat_T, _ = build_feature_matrix(df_natural, fs_T_opx, use_liq=False)
    X_nat_P, _ = build_feature_matrix(df_natural, fs_P_opx, use_liq=False)

    df_natural['ood_score'] = iso.score_samples(X_nat_P)
    df_natural['is_outlier'] = iso.predict(X_nat_P) == -1

    df_natural['T_pred'] = qrf_T_opx.predict(X_nat_T, quantiles=[0.5]).flatten()
    df_natural['T_lo'] = qrf_T_opx.predict(X_nat_T, quantiles=[0.16]).flatten()
    df_natural['T_hi'] = qrf_T_opx.predict(X_nat_T, quantiles=[0.84]).flatten()
    df_natural['P_pred'] = qrf_P_opx.predict(X_nat_P, quantiles=[0.5]).flatten()
    df_natural['P_lo'] = qrf_P_opx.predict(X_nat_P, quantiles=[0.16]).flatten()
    df_natural['P_hi'] = qrf_P_opx.predict(X_nat_P, quantiles=[0.84]).flatten()

    df_natural['T_CV'] = (df_natural['T_hi'] - df_natural['T_lo']) / np.abs(df_natural['T_pred']).clip(lower=1.0)
    df_natural['P_CV'] = (df_natural['P_hi'] - df_natural['P_lo']) / np.abs(df_natural['P_pred']).clip(lower=1.0)

    print(f'IsolationForest flagged {int(df_natural["is_outlier"].sum())}/{len(df_natural)} as outliers')
    df_filtered = df_natural[~df_natural['is_outlier']].copy()
    print(f'after OOD filter: {len(df_filtered)}')
    df_filtered = df_filtered[df_filtered['P_CV'] < 0.3].copy()
    print(f'after CV<0.3 filter: {len(df_filtered)}')

    for cv_cutoff in [0.2, 0.3, 0.5]:
        n = (df_natural[~df_natural['is_outlier']]['P_CV'] < cv_cutoff).sum()
        print(f'  CV<{cv_cutoff}: {int(n)}')
else:
    df_filtered = pd.DataFrame()
    print('No natural samples to score; df_filtered is empty.')

IsolationForest flagged 217/3145 as outliers
after OOD filter: 2928
after CV<0.3 filter: 1
  CV<0.2: 0
  CV<0.3: 1
  CV<0.5: 9


In [11]:
# Save predictions and run physical-plausibility checks
out_path_all = RESULTS / 'nb08_natural_predictions_all.csv'
out_path_filt = RESULTS / 'nb08_natural_predictions_filtered.csv'

if len(df_natural) > 0:
    df_natural.to_csv(out_path_all, index=False)
    df_filtered.to_csv(out_path_filt, index=False)
    if len(df_filtered) > 0:
        assert df_filtered['P_pred'].max() < 80, f"P>80 kbar implausible: {df_filtered['P_pred'].max()}"
        assert df_filtered['T_pred'].max() < 1700, f"T>1700 C implausible: {df_filtered['T_pred'].max()}"
        assert df_filtered['T_pred'].min() > 400, f"T<400 C implausible: {df_filtered['T_pred'].min()}"
        print('Plausibility checks: PASS')
else:
    note_df = pd.DataFrame([{
        'note': 'Lin 2023 supplementary tables (Tables S1-S6) contain no orthopyroxene major-element data. '
                'Provide a CSV with opx SiO2/Al2O3/FeO_total/MgO/CaO at data/natural/ to populate this table.'
    }])
    note_df.to_csv(out_path_all, index=False)
    note_df.to_csv(out_path_filt, index=False)
    print('Saved empty natural-prediction tables with explanatory header.')

print('\n=== PHASE 8 COMPLETE ===')

Plausibility checks: PASS

=== PHASE 8 COMPLETE ===
